# 01 — Case Classification and Case-Control Matching (SGA, FGR and AGA)

**Manuscript:** Associations between gut microbiota diversity in early and late pregnancy and small for gestational age or growth restricted births: a nested case-control study

**Purpose:** Classifies births as small for gestational age (SGA), fetal growth restricted (FGR) or appropriate for gestational age (AGA) using the Swedish population-based fetal growth reference chart
A common pool of AGA controls is shared between the two comparisons. Mothers with [exclusions, e.g. multiple pregnancy, missing TP1/TP2 samples] are excluded.

**Produces:** Analysis-ready metadata with case/control status and matched-set IDs (input for scripts 02 onwards) and Figure 1 

**Programmer:** Anusha Antony

# Load data

In [ ]:
library(dplyr)
library(tidyverse)
library(readxl)
library(openxlsx)
library(writexl)
library(vegan)

In [ ]:
base_path <- "data"

In [ ]:
ref_file <- read_excel(file.path(base_path("data"))
head(ref_file)

In [ ]:
all_data <- read.table(file.path(base_path("data"),sep = ",", header = TRUE)
dim(all_data)

taxa_tab <- read.table(file.path(base_path("data",sep ="\t", header = TRUE,check.names = FALSE, quote='"')
#head(taxa_tab)
dim(taxa_tab)

In [ ]:
head(taxa_tab)

In [ ]:
class(all_data$kit1.faecal_sample.datetime_taken)

In [ ]:
grep("kit2", names(all_data), value = TRUE)

# Exclusion criteria and birth weight classification

In [ ]:
kit <- as.Date(all_data$kit1.faecal_sample.datetime_taken)
#kit 
q1 <- as.Date(all_data$Q1_Datum)
#q1
kit1_Q1_date <- as.integer(difftime(kit,q1, units = "week"))
#kit1_Q1_date
sampling_week <- all_data$Q1_X17_Pregnancy_week + kit1_Q1_date
#sum(is.na(sampling_week))
all_data$update_pregnancy_week_of_sampling<- sampling_week
all_data$update_pregnancy_week_of_sampling

In [ ]:
all_data <- all_data %>% filter(!(is.na(kit1.faecal_sample.datetime_taken) | is.na(kit1.faecal_sample.barcode)))
dim(all_data)


In [ ]:
kit_1_missing <- all_data %>% filter(!(is.na(kit1.faecal_sample.datetime_taken) | is.na(kit1.faecal_sample.barcode)))
dim(all_data)

In [ ]:
table(all_data$Q3_X8_One_or_more_child)

In [ ]:
sum(is.na(all_data$Q3_X8_One_or_more_child))

In [ ]:
#remove the twins:  Ett barn - A child, Tvillingar - Twins, Q3_X8..Födde.du.ett.barn.eller.fler
table(all_data$Q3_X8_One_or_more_child)

all_no_twin <- all_data %>% filter(all_data$Q3_X8_One_or_more_child == "Ett barn")
dim(all_no_twin)
head(all_no_twin)

In [ ]:
#girl and boy
#Flicka - girl
#Pojke - boy
table(all_no_twin$Q3_X10_Boy_or_girl)

In [ ]:
#remove the rows with Na in the required columns
filt_all_data1 <- all_no_twin  %>% filter(!(is.na(all_no_twin$GL_v)))
dim(filt_all_data1)
all_flt_2 <- filt_all_data1 %>% filter(!(is.na(Q3_X10_Boy_or_girl)))
dim(all_flt_2)
all_flt_3 <- all_flt_2 %>% filter(!(is.na(Birthweight_checked)))
dim(all_flt_3)
#head(all_flt_3)

In [ ]:
all_flt_3$GL_d

In [ ]:
flt_week <- all_flt_3 %>% filter(all_flt_3$pregnancy_week_of_sampling_variable <= 20)
dim(flt_week)

In [ ]:
find_weight_class <- function(GW, GD, weight, SN, ref) {
  if (as.numeric(GD) > 0) {
    GW <- as.numeric(GW) + 1
  }

  # Find the row in the ref file corresponding to the Studienummer
  row_index <- match(SN, ref$ID)

  if (!is.na(row_index)) {
    if (ref$percentil[row_index] < 2.5) {
      return("tiny_SGA")
    } else if (ref$percentil[row_index] >= 2.5 & ref$percentil[row_index] < 10) {
      return("SGA")
    } else if (ref$percentil[row_index] >= 10 & ref$percentil[row_index] <= 90) {
      return("AGA")
    } else if (ref$percentil[row_index] >= 97.5) {
      return("LGA")
    }
      else{
          return("NA")
    }
  }

  
}
#NA indicates the babies between 91 to 97. 

In [ ]:
weight_classes<-vector(length=nrow(flt_week))
for(i in 1:length(weight_classes)){
    weight_classes[i]<-find_weight_class(flt_week$GL_v[i], flt_week$GL_d[i],
                                        flt_week$Birthweight_checked[i], 
                                         flt_week$Studienummer[i],ref_file )
}

In [ ]:
table(weight_classes)

In [ ]:
length(setdiff(flt_week$Studienummer,ref_file$ID))

In [ ]:
lga_babies <- flt_week$Studienummer[which(weight_classes == "LGA")]
length(lga_babies)
aga_babies <- flt_week$Studienummer[which(weight_classes == "AGA")]
#aga_babies
length(aga_babies)
sga_babies <- flt_week$Studienummer[which(weight_classes == "SGA")]
length(sga_babies)
tiny_sga_babies <- flt_week$Studienummer[which(weight_classes == "tiny_SGA")]
length(tiny_sga_babies)

### Check with sequenced data

In [ ]:
sga_kit_1 <- all_data %>% filter(Studienummer %in% sga_babies) %>% pull (kit1.faecal_sample.barcode)
length(sga_kit_1)

In [ ]:
fgr_kit_1 <- all_data %>% filter(Studienummer %in% tiny_sga_babies) %>% pull (kit1.faecal_sample.barcode)
length(fgr_kit_1)

In [ ]:
sample_list <- read.csv(file.path(base_path("data"))
head(sample_list)
dim(sample_list)

In [ ]:
length(intersect(sample_list$sample_id, sga_kit_1))

In [ ]:
length(intersect(sample_list$sample_id, fgr_kit_1))

#### Continue

In [ ]:
lga_babies

In [ ]:
dim(flt_week %>% filter(Studienummer %in% aga_babies & flt_week$double_participation == "Yes"))
dim(flt_week %>% filter(Studienummer %in% lga_babies & flt_week$double_participation == "Yes"))
dim(flt_week %>% filter(Studienummer %in% sga_babies & flt_week$double_participation == "Yes"))
dim(flt_week %>% filter(Studienummer %in% tiny_sga_babies & flt_week$double_participation == "Yes"))

In [ ]:
sum(is.na(flt_week$Studienummer))

## Make a list of Studynumbers/sample id's that has to be removed before case control match


### Remove the studynumbers of the second pregnancy. Include only thier first pregnancy information. 

In [ ]:
double_preg_all <- flt_week %>%
  filter(
    (Studienummer %in% tiny_sga_babies & double_participation == "Yes") |
    (Studienummer %in% aga_babies & double_participation == "Yes") |
    (Studienummer %in% sga_babies & double_participation == "Yes")
  ) %>% select(Studienummer, Q1_Personnummer, kit1.faecal_sample.barcode,Q1_Datum)
dim(double_preg_all)
head(double_preg_all)

In [ ]:
double_preg_taxa <- taxa_tab[, colnames(taxa_tab) %in% double_preg_all$kit1.faecal_sample.barcode]
dim(double_preg_taxa)
look_into_it <- colnames(double_preg_taxa)
look_into_it

In [ ]:
search_for_it <- double_preg_all %>% filter(kit1.faecal_sample.barcode %in% look_into_it)
#search_for_it

In [ ]:
# Convert character to POSIXct (date-time) format
search_for_it$Q1_Datum <- as.POSIXct(search_for_it$Q1_Datum, format = "%Y-%m-%d %H:%M:%S")

# Remove the time part and keep only the date
search_for_it$Q1_Datum <- as.Date(search_for_it$Q1_Datum)

In [ ]:
extract_first_sec_preg <- function(data_df) {
  sorted_data <- data_df %>%
    arrange(Q1_Personnummer, Q1_Datum)
  
  # Extract the first and second pregnancies per group
  pregnancies <- sorted_data %>%
    group_by(Q1_Personnummer) %>%
    mutate(pregnancy_order = row_number()) %>%
    filter(pregnancy_order <= 2) %>%
    ungroup()
  
  # Separate into first and second pregnancies
  first_preg <- pregnancies %>%
    filter(pregnancy_order == 1)
  
  second_preg <- pregnancies %>%
    filter(pregnancy_order == 2)
  
  return(list(first_preg = first_preg, second_preg = second_preg))
}



In [ ]:
find_who_twice <- extract_first_sec_preg(search_for_it)
first <- find_who_twice$first_preg
dim(first)
second <- find_who_twice$second_preg
dim(second)
head(second)
#remove all the second pregnancy samples from the list. 

In [ ]:
double_peg <- second$Studienummer
double_peg

In [ ]:
length(intersect(double_peg,aga_babies))
length(intersect(double_peg,sga_babies))
length(intersect(double_peg,tiny_sga_babies))

### Remove the Studynumbers for those whose microbiome data is absent in the list

In [ ]:
yes_taxa <- colnames(taxa_tab)
length(yes_taxa )
all_id_i_need <- all_data %>% filter(Studienummer %in% aga_babies | Studienummer %in% tiny_sga_babies |Studienummer %in% sga_babies ) %>% pull(kit1.faecal_sample.barcode)
length(all_id_i_need)
length(intersect(yes_taxa , all_id_i_need))
remove_id <- setdiff(all_id_i_need,yes_taxa)
length(remove_id)
remove_id_sn <- all_data %>% filter(kit1.faecal_sample.barcode %in% remove_id) %>% pull(Studienummer)
length(remove_id_sn)
#remove the 'remove_id_sn' from casr control list

### Remove due to low richness (less than 120)

In [ ]:
library(vegan)

In [ ]:
#Add other_taxa row to make all the columns same constant value
new_row <- c()
tot <- colSums(taxa_tab)
for(i in tot){
    if(i < 100) {
        new_row <- c(new_row,100 - i)
    } else if( i >=100 ){
        new_row <- c(new_row,0)
    }
}

#add this vector as row to the taxa table

new_row <- setNames(new_row, names(taxa_tab))
#new_row
new_row2 <- as.data.frame(t(new_row))
#new_row2
rownames(new_row2) <- "Other_taxa"
best_filt_new <- rbind(taxa_tab,new_row2) 
#colSums(best_filt_new )
#now the colSums == 100

filt_new_arrange <- as.data.frame(best_filt_new)
head(filt_new_arrange)
dim(filt_new_arrange)

In [ ]:
taxa_richness <- apply(filt_new_arrange, 2, specnumber)
less_richness <- which(taxa_richness < 120)
id_less_richness <- names(taxa_richness)[less_richness]
id_less_richness

In [ ]:
remove_id_richness <- all_data %>% filter(kit1.faecal_sample.barcode %in% id_less_richness) %>% pull(Studienummer)
remove_id_richness
length(remove_id_richness)

In [ ]:
length(intersect(remove_id_richness,aga_babies))
length(intersect(remove_id_richness,sga_babies))
length(intersect(remove_id_richness,tiny_sga_babies))

In [ ]:
#There was one participant with age 17. so remove it.
age_less <- c("S87c5ff3")

In [ ]:
length(intersect(age_less,aga_babies))
length(intersect(age_less,sga_babies))
length(intersect(age_less,tiny_sga_babies))

### collect all the Study numbers to drop

In [ ]:
drop_these <- c(age_less,remove_id_richness,remove_id_sn,second$Studienummer)  

In [ ]:
drop_tiny_sga_sn <- intersect(tiny_sga_babies,drop_these)
#drop_tiny_sga_sn

drop_sga_sn <- intersect(sga_babies,drop_these) 
##drop_sga_sn

drop_aga_sn <- intersect(aga_babies,drop_these) 
#drop_aga_sn
length(drop_aga_sn)

In [ ]:
final_tiny_sga <- setdiff(tiny_sga_babies, drop_tiny_sga_sn)
length(final_tiny_sga)
final_sga <- setdiff(sga_babies, drop_sga_sn)
length(final_sga)
final_aga <- setdiff(aga_babies, drop_aga_sn)
length(final_aga)

#### Due to down's syndrome we need to remove the 2 control group
I am doing this on 13th of march, to replace the 2 control groups with the other

In [ ]:
small_control_data_generated <- read.table("/ceph/projects/010_SweMaMi/anusha.antony/Anu_SweMaMi/project_neonate_size/SGA_new/SGA_solve_outliers_250102/data/meta/Small_control_matchit_171_samecontrol_250108.tsv",header = TRUE)
head(small_control_data_generated)
control_group <- c(small_control_data_generated$Control1, small_control_data_generated$Control2)

In [ ]:
setdiff(control_group, final_filt_aga)
length(intersect(control_group, final_filt_aga))
length(setdiff(final_filt_aga, control_group))
find_control <- setdiff(final_filt_aga, control_group)
#find_control 

In [ ]:
replace_S07a975a_16 <- all_data %>% filter(Studienummer %in% find_control) %>%
filter(pregnancy_week_of_sampling_variable == 16) %>% pull(Studienummer)
length(replace_S07a975a_16)

In [ ]:
replace_S92b4604_17 <- all_data %>% filter(Studienummer %in% find_control) %>%
filter(pregnancy_week_of_sampling_variable == 17) %>% pull(Studienummer)
length(replace_S92b4604_17)

In [ ]:
#check whether they have down's syndrome
barndiag <- read.table("/ceph/projects/010_SweMaMi/metadata/SocialStyrelsen_5036-2022/SOS_BARN_BARNETS_DIAGNOSER_FO.txt",
                      sep="\t", header = TRUE)
head(barndiag)

In [ ]:
gravreg <- read.table("/ceph/projects/010_SweMaMi/metadata/SocialStyrelsen_5036-2022/UT_R_MFR_5036_2022.txt",
                      sep="\t", header = TRUE)
head(gravreg)

In [ ]:
ids <- read.table("/ceph/projects/010_SweMaMi/metadata/SocialStyrelsen_5036-2022/SOS_POP_GR.txt",
                      sep="\t", header = TRUE)
dim(ids)
head(ids)[,c(1, 2, 15, 16)]

In [ ]:
ids_16 <- ids[which(ids$pnr %in% replace_S07a975a_16),]
dim(ids_16)

In [ ]:
ids_17 <- ids[which(ids$pnr %in% replace_S92b4604_17),]
dim(ids_17)

In [ ]:
replace_S07a975a_16
length(replace_S07a975a_16)

In [ ]:
replace_S92b4604_17
length(replace_S92b4604_17)

In [ ]:
sum(is.na(all_data$kit1.faecal_sample.datetime_taken))
missing_samples <- all_data %>% filter(is.na(all_data$kit2.faecal_sample.datetime_taken)) %>% pull(Studienummer)


In [ ]:
remove_16_1 <- intersect(replace_S07a975a_16,missing_samples) 
remove_17_1 <- intersect(replace_S92b4604_17,missing_samples ) 
selected_16 <- setdiff(replace_S07a975a_16,remove_16_1) 
length(selected_16)
selected_17 <- setdiff(replace_S92b4604_17,remove_17_1) 
length(selected_17)

In [ ]:
all_data$kit1.faecal_sample.barcode

In [ ]:
kit1_yes <- all_data %>% filter(Studienummer %in% selected_16 | Studienummer %in% selected_17) %>% pull(kit1.faecal_sample.barcode)
kit2_yes <- all_data %>% filter(Studienummer %in% selected_16 | Studienummer %in% selected_17) %>% pull(kit2.faecal_sample.barcode)

In [ ]:
all_kit <- c(kit1_yes, kit2_yes)
length(all_kit)

In [ ]:
length(intersect(colnames(taxa_tab),all_kit) )
present_id <- intersect(colnames(taxa_tab),all_kit)
missing_kit_samples <- setdiff(all_kit,present_id)
find_sn_missing_kit <- all_data %>% filter(kit1.faecal_sample.barcode %in% missing_kit_samples |kit2.faecal_sample.barcode %in% missing_kit_samples) %>% pull(Studienummer)
length(find_sn_missing_kit)

In [ ]:
selected_16_final <- setdiff(selected_16, find_sn_missing_kit)
length(selected_16_final)
selected_17_final <- setdiff(selected_17, find_sn_missing_kit)
length(selected_17_final)

In [ ]:
selected_16_final

In [ ]:
selected_17_final

In [ ]:
#So none these samples have down's syndrome. So consider any of them
#I am selecting: S3196496 to replace S07a975a and S114eb54 to replace S92b4604 controls

### continue

In [ ]:
tiny_sga_bc <- all_data %>% filter(Studienummer %in% final_tiny_sga) %>% pull(kit1.faecal_sample.barcode)
#tiny_sga_bc 
final_filt_tiny_sga <- all_data %>% filter(kit1.faecal_sample.barcode %in% colnames(taxa_tab[, colnames(taxa_tab) %in% tiny_sga_bc ])) %>% pull(Studienummer)
length(final_filt_tiny_sga)

In [ ]:
large_sga_bc <- all_data %>% filter(Studienummer %in% final_sga) %>% pull(kit1.faecal_sample.barcode)
#tiny_sga_bc 
final_filt_sga <- all_data %>% filter(kit1.faecal_sample.barcode %in% colnames(taxa_tab[, colnames(taxa_tab) %in% large_sga_bc ])) %>% pull(Studienummer)
length(final_filt_sga)
#All the SGA data available
#my_SGA_sn <- "/ceph/projects/010_SweMaMi/anusha.antony/Anu_SweMaMi/project_neonate_size/SGA_new/data/fecal/TP1/table/SGA_small_taxa_present.txt"
#my_sga_small_sn <- as.character(final_filt_sga)
#writeLines(my_sga_small_sn,my_SGA_sn)

In [ ]:
aga_bc <- all_data %>% filter(Studienummer %in% final_aga) %>% pull(kit1.faecal_sample.barcode)
final_filt_aga <- all_data %>% filter(kit1.faecal_sample.barcode %in% colnames(taxa_tab[, colnames(taxa_tab) %in% aga_bc ])) %>% pull(Studienummer)
length(final_filt_aga)
#aga_with_taxa

#All the SGA data available
#aga_with_taxa_path <- "/ceph/projects/010_SweMaMi/anusha.antony/Anu_SweMaMi/project_neonate_size/SGA_new/data/fecal/TP1/table/AGA_taxa_present.txt"
#aga_with_taxa <- as.character(final_filt_aga)
#writeLines(aga_with_taxa ,aga_with_taxa_path)

In [ ]:
all_sn_include <- final_filt_aga
length(all_sn_include)

new_all_data <- all_data %>% filter(Studienummer %in% all_sn_include )
dim(new_all_data )

In [ ]:
sum(is.na(new_all_data$kit2.faecal_sample.barcode))

In [ ]:
two_tp1_data_present <- new_all_data %>% filter(!(is.na(kit2.faecal_sample.barcode))) 
dim(two_tp1_data_present)
head(two_tp1_data_present)

In [ ]:
two_tp1_data_present$Q1_Personnummer

In [ ]:
ids_with_ds <- ids[which(ids$pnr %in% two_tp1_data_present$Q1_Personnummer),]
dim(ids_with_ds)

In [ ]:
ids <- ids[,c(16, 1)]
barndiag <- merge(ids, barndiag, by = "glopnr")
dim(barndiag)

In [ ]:
remove_it <- barndiag[which(barndiag$diagnoskod_barnets_diagnoser == "Q909"),]
dim(remove_it)
remove_it

In [ ]:
sn_to_remove <- two_tp1_data_present %>% filter(Q1_Personnummer %in% remove_it$pnr) %>% pull(Studienummer)
sn_to_remove  

In [ ]:
tp1_tp2_aga_present_no_ds <- two_tp1_data_present %>% filter(!(Studienummer %in% sn_to_remove))
dim(tp1_tp2_aga_present_no_ds)

In [ ]:
all_kit1_kit2 <- c(tp1_tp2_aga_present_no_ds$kit1.faecal_sample.barcode, tp1_tp2_aga_present_no_ds$kit2.faecal_sample.barcode)
length(all_kit1_kit2)

In [ ]:
missing_taxa_tab <- setdiff(all_kit1_kit2, colnames(taxa_tab))
length(missing_taxa_tab)

In [ ]:
drop_sn_from_taxa_tab <- tp1_tp2_aga_present_no_ds %>% filter(kit1.faecal_sample.barcode %in% missing_taxa_tab | kit2.faecal_sample.barcode %in% missing_taxa_tab) %>% pull(Studienummer)
length(drop_sn_from_taxa_tab)

In [ ]:
final_aga_df <- tp1_tp2_aga_present_no_ds %>% filter(!(Studienummer %in% drop_sn_from_taxa_tab))
dim(final_aga_df)

In [ ]:
final_aga_filt_set <- final_aga_df$Studienummer


In [ ]:
final_filt_sga

In [ ]:
#remove down's syndrome cases from tiny and small. S8d9a376 in small cases.
length(final_filt_sga)
final_filt_sga_set <- final_filt_sga[final_filt_sga != "S8d9a376"]

length(final_filt_sga_set)

In [ ]:
all_final_sn_include <- c(final_filt_sga_set, final_aga_filt_set,final_filt_tiny_sga) 
length(all_final_sn_include)

In [ ]:
refined_set <- flt_week %>% filter(Studienummer %in% all_final_sn_include)
dim(refined_set)
#head(refined_set)

In [ ]:
#Now I removed all the problematic studynumbers from my the cohort. Now let's move to the case control grouping

## Case- control grouping - method-1

In [ ]:
raw_data <- refined_set %>% select(Studienummer,update_pregnancy_week_of_sampling,kit1.faecal_sample.barcode ) %>% filter(!(update_pregnancy_week_of_sampling < 6 | update_pregnancy_week_of_sampling >20))
dim(raw_data)
sum(is.na(raw_data))

In [ ]:
dim(raw_data %>% filter(kit1.faecal_sample.barcode %in% colnames(taxa_tab)))

In [ ]:
new_raw_data <- refined_set %>% select(Studienummer,update_pregnancy_week_of_sampling) %>% filter(!(update_pregnancy_week_of_sampling < 6 | update_pregnancy_week_of_sampling >20))
dim(new_raw_data )
sum(is.na(new_raw_data))

In [ ]:
find_controls_3 <- function(data, case_ids, control_ids, large_SGA_ids) {
    used_ids <- c()  
    no_match <- c()
    output <- matrix(nrow = length(case_ids), ncol = 4)
    
    for (i in 1:length(case_ids)) {
        id <- case_ids[i]
        
       
        if (id %in% used_ids) {
            print(paste("Case", id, "is already used. Skipping."))
            next
        }
        
        myrow <- which(data$Studienummer == id)
        case_week <- data$update_pregnancy_week_of_sampling[myrow]
        
        # Function to find controls within the week range
        find_controls_in_group <- function(group_ids) {
            week_gap <- abs(data$update_pregnancy_week_of_sampling[data$Studienummer %in% group_ids] - case_week)
            potential_controls <- group_ids[week_gap <= 1]
            # Ensure controls haven't been used yet (in any column)
            setdiff(potential_controls, used_ids)
        }
        
        # Find potential controls in each group
        controls_normal <- find_controls_in_group(control_ids)
        controls_large_SGA <- find_controls_in_group(large_SGA_ids)
        
       
        selected_controls <- c(NA, NA, NA)
        
        # Choose Control 1 and Control 2 from the normal group if available
        if (length(controls_normal) > 0) {
            selected_controls[1] <- controls_normal[1]
            controls_normal <- setdiff(controls_normal, selected_controls[1])
            
            if (length(controls_normal) > 0) {
                selected_controls[2] <- controls_normal[1]
            }
        }
        
       
        if (length(controls_large_SGA) > 0) {
            selected_controls[3] <- controls_large_SGA[1]
        }
        
        # Check if any controls are missing
        if (any(is.na(selected_controls))) {
            print(paste(c("Case", id, "has", sum(!is.na(selected_controls)), "unique controls"), collapse = " "))
            no_match <- c(no_match, id)
        }
        
        output[i,] <- c(id, selected_controls)
        used_ids <- c(used_ids, id, selected_controls[!is.na(selected_controls)])
    }
    
    colnames(output) <- c("Case_1_tiny", "Control1", "Control2", "Case_2_small")
    return(list(match = output, fail = no_match))
}

In [ ]:
sga_match_3<-find_controls_3(data = new_raw_data, case_ids = final_filt_tiny_sga,control_ids = final_filt_aga,large_SGA_ids = final_filt_sga )
cs_ct_3 <- sga_match_3$match
#cs_ct
cs_ct_3<- as.data.frame(sga_match_3$match)
head(cs_ct_3)
dim(cs_ct_3)

In [ ]:
week_tiny <- new_raw_data %>% filter(Studienummer %in% cs_ct_3$Case_1_tiny) %>% pull(update_pregnancy_week_of_sampling)
week_control1 <- new_raw_data %>% filter(Studienummer %in% cs_ct_3$Control1) %>% pull(update_pregnancy_week_of_sampling)
week_control2 <- new_raw_data %>% filter(Studienummer %in% cs_ct_3$Control2) %>% pull(update_pregnancy_week_of_sampling)
week_small <- new_raw_data %>% filter(Studienummer %in% cs_ct_3$Case_2_small) %>% pull(update_pregnancy_week_of_sampling)

In [ ]:
new_ct_cs_week <- cbind(cs_ct_3, week_tiny,week_control1 ,week_control2,week_small)
head(new_ct_cs_week)

In [ ]:
# There is no excat match. Hence, it is a problem. So, it's better use the matchit function output.

In [ ]:
intersect(cs_ct_3$Case_1_tiny, cs_ct_3$Case_2_small)

In [ ]:
all_sn <- c(cs_ct_3$Case_1_tiny, cs_ct_3$Control1, cs_ct_3$Control2, cs_ct_3$Case_2_small)
length(unique(all_sn))

In [ ]:
write.table(cs_ct_3,"/ceph/projects/010_SweMaMi/anusha.antony/Anu_SweMaMi/project_neonate_size/SGA_new/data/fecal/TP1/table/case_control_3groups.tsv", sep = "\t", row.names = FALSE)

In [ ]:
sum(is.na(cs_ct_3))

## Case-Control- method2

In [ ]:
library("MatchIt")
library(dplyr)
library(tidyverse)
library(readxl)
library(data.table)

In [ ]:
raw_data <- refined_set %>% select(Studienummer,update_pregnancy_week_of_sampling,kit1.faecal_sample.barcode ) %>% filter(!(update_pregnancy_week_of_sampling < 6 | update_pregnancy_week_of_sampling >20))
dim(raw_data)
sum(is.na(raw_data))

In [ ]:
dim(taxa_tab[, colnames(taxa_tab) %in% raw_data$kit1.faecal_sample.barcode])

In [ ]:
raw_data_filt <- raw_data %>% filter(kit1.faecal_sample.barcode %in% colnames(taxa_tab))
dim(raw_data_filt)

In [ ]:
new_raw_data <- raw_data_filt %>% select(Studienummer,update_pregnancy_week_of_sampling) %>% filter(!(update_pregnancy_week_of_sampling < 6 | update_pregnancy_week_of_sampling >20))
dim(new_raw_data )
sum(is.na(new_raw_data))

### Case-FGR

In [ ]:
new_raw_data$group <- ifelse(new_raw_data$Studienummer %in% final_tiny_sga,"Case_tiny","Control")

In [ ]:
head(new_raw_data)

In [ ]:
# Ensure 'group' is a factor with 'case' as the treated group
new_raw_data$group <- factor(new_raw_data$group, levels = c("Control", "Case_tiny"))

In [ ]:
table(new_raw_data$group)

In [ ]:
# Perform matching
set.seed(42) 
m.out <- matchit(group ~ update_pregnancy_week_of_sampling, 
                 data = new_raw_data, 
                 method = "nearest",ratio = 2)

In [ ]:
matched_data <- match.data(m.out)
head(matched_data )

In [ ]:
cases <- matched_data %>% filter(group == "Case_tiny")
controls <- matched_data %>% filter(group == "Control")
dim(cases)
dim(controls)

In [ ]:
close_ct_df <- data.frame(Case = character(),
                                  Control1 = character(),
                                  Control2 = character(),
                                  stringsAsFactors = FALSE)
head(close_ct_df)

In [ ]:
used_controls <- c()
for (i in 1:nrow(cases)) {
  case_row <- cases[i, ]
  free_controls <- controls %>% 
    filter(!(Studienummer %in% used_controls))
  
  if (nrow(free_controls) < 2) {
    message("Not enough available controls for case: ", case_row$Studienummer)
    break
  }
    free_controls$case_distance <- abs(free_controls$distance - case_row$distance)
  
  closest_controls <- free_controls %>%
    arrange(case_distance) %>%
    slice(1:2)

  close_ct_df  <- rbind(close_ct_df , data.frame(
    Case = case_row$Studienummer,
    Control1 = closest_controls$Studienummer[1],
    Control2 = closest_controls$Studienummer[2],
    stringsAsFactors = FALSE
  ))
  
  used_controls <- c(used_controls, closest_controls$Studienummer)
}
head(close_ct_df )
dim(close_ct_df )

In [ ]:
case_week <- matched_data[matched_data$Studienummer %in% close_ct_df$Case,]
new_case_week <- case_week%>% select(Studienummer,update_pregnancy_week_of_sampling)
# Rename multiple columns for old to new
setnames(new_case_week, old = c("Studienummer","update_pregnancy_week_of_sampling"), 
         new = c("Case","Case_week"))
head(new_case_week)

In [ ]:
control1_week <- matched_data[matched_data$Studienummer %in% close_ct_df$Control1,]
control1_week <- control1_week %>% select(Studienummer,update_pregnancy_week_of_sampling)
# Rename multiple columns for old to new
setnames(control1_week, old = c("Studienummer","update_pregnancy_week_of_sampling"), 
         new = c("Control1","Control1_week"))
head(control1_week )

In [ ]:
control2_week <- matched_data[matched_data$Studienummer %in% close_ct_df$Control2,]
control2_week <- control2_week %>% select(Studienummer,update_pregnancy_week_of_sampling)
# Rename multiple columns for old to new
setnames(control2_week, old = c("Studienummer","update_pregnancy_week_of_sampling"), 
         new = c("Control2","Control2_week"))
head(control2_week)

In [ ]:
merge_1 <- merge(close_ct_df,new_case_week, by = "Case", all.x = TRUE) 
head(merge_1)

In [ ]:
merge_2 <- merge(merge_1,control1_week, by = "Control1", all.x = TRUE) 
head(merge_2)

In [ ]:
merge_3 <- merge(merge_2,control2_week, by = "Control2", all.x = TRUE) 
head(merge_3)

In [ ]:
only_week <- merge_3 %>% select(Case_week,Control1_week, Control2_week)
head(only_week)
#only_week

In [ ]:
which(duplicated(close_ct_df$Case, close_ct_df$Control1) == TRUE)
which(duplicated(close_ct_df$Case, close_ct_df$Control2) == TRUE)
which(duplicated(close_ct_df$Control1, close_ct_df$Control2) == TRUE)

In [ ]:
tiny_control <- c(close_ct_df$Control1,close_ct_df$Control2) 
length(tiny_control)

### Case-SGA

In [ ]:
new_raw_data$group <- ifelse(new_raw_data$Studienummer %in% final_sga,"Case_small","Control")
head(new_raw_data)

In [ ]:
#new_raw_data

In [ ]:
# Ensure 'group' is a factor with 'case' as the treated group
new_raw_data$group <- factor(new_raw_data$group, levels = c("Control", "Case_small"))

In [ ]:
table(new_raw_data$group)

In [ ]:
# Perform matching
set.seed(42) 
m.out <- matchit(group ~ update_pregnancy_week_of_sampling, 
                 data = new_raw_data, 
                 method = "nearest",ratio = 2)

matched_data <- match.data(m.out)
head(matched_data )

In [ ]:
cases <- matched_data %>% filter(group == "Case_small")
controls <- matched_data %>% filter(group == "Control")
dim(cases)
dim(controls)

In [ ]:
close_ct_df_1 <- data.frame(Case = character(),
                                  Control1 = character(),
                                  Control2 = character(),
                                  stringsAsFactors = FALSE)
head(close_ct_df_1)

In [ ]:
used_controls <- c()
for (i in 1:nrow(cases)) {
  case_row <- cases[i, ]
  free_controls <- controls %>% 
    filter(!(Studienummer %in% used_controls))
  
  if (nrow(free_controls) < 2) {
    message("Not enough available controls for case: ", case_row$Studienummer)
    break
  }
    free_controls$case_distance <- abs(free_controls$distance - case_row$distance)
  
  closest_controls <- free_controls %>%
    arrange(case_distance) %>%
    slice(1:2)

  close_ct_df_1  <- rbind(close_ct_df_1 , data.frame(
    Case = case_row$Studienummer,
    Control1 = closest_controls$Studienummer[1],
    Control2 = closest_controls$Studienummer[2],
    stringsAsFactors = FALSE
  ))
  
  used_controls <- c(used_controls, closest_controls$Studienummer)
}
head(close_ct_df_1 )
dim(close_ct_df_1 )

In [ ]:
small_control <- c(close_ct_df_1$Control1, close_ct_df_1$Control2)

In [ ]:
length(intersect(small_control,tiny_control ))


In [ ]:
intersect(close_ct_df_1$Control1, close_ct_df$Control1)

In [ ]:
intersect(close_ct_df_1$Control2, close_ct_df$Control2)

In [ ]:
intersect(close_ct_df_1$Control2, close_ct_df$Control1)

## Case Control group-FGR,SGA, control1, control2 using matchit function
this data was used further

In [ ]:
library("MatchIt")
library(dplyr)
library(tidyverse)
library(readxl)
library(data.table)

In [ ]:
packageVersion("MatchIt")

In [ ]:
raw_data <- refined_set %>% select(Studienummer,update_pregnancy_week_of_sampling,kit1.faecal_sample.barcode ) %>% filter(!(update_pregnancy_week_of_sampling < 6 | update_pregnancy_week_of_sampling >20))
dim(raw_data)
sum(is.na(raw_data))

In [ ]:
raw_data_filt <- raw_data %>% filter(kit1.faecal_sample.barcode %in% colnames(taxa_tab))
dim(raw_data_filt)

In [ ]:
new_raw_data <- raw_data_filt %>% select(Studienummer,update_pregnancy_week_of_sampling) %>% filter(!(update_pregnancy_week_of_sampling < 6 | update_pregnancy_week_of_sampling >20))
dim(new_raw_data )
sum(is.na(new_raw_data))

In [ ]:
new_raw_data$group <- ifelse(new_raw_data$Studienummer %in% final_filt_tiny_sga,"Case_tiny",
                             ifelse(new_raw_data$Studienummer %in% final_filt_sga_set,"Case_small", "Control"))
head(new_raw_data)

#### Tiny

In [ ]:
# Ensure 'group' is a factor with 'case' as the treated group
new_raw_data$group <- factor(new_raw_data$group, levels = c("Control","Case_tiny", "Case_small"))

In [ ]:
table(new_raw_data$group)

In [ ]:
# First stage: Match Control to Case_tiny
data_tiny_control <- new_raw_data %>% filter(group %in% c("Case_tiny", "Control"))
head(data_tiny_control)
table(data_tiny_control$group)

In [ ]:
data_tiny_control <- data_tiny_control %>% droplevels()
table(data_tiny_control$group)

In [ ]:
# Perform matching
set.seed(42) 
m.out_tiny_control <- matchit(group ~ update_pregnancy_week_of_sampling, 
                              data = data_tiny_control, 
                              method = "nearest", 
                              ratio = 2)
matched_data_tiny <- match.data(m.out_tiny_control)
head(matched_data_tiny)
dim(matched_data_tiny)

In [ ]:
tiny_control_list <- matched_data_tiny %>% filter(group == "Control")
dim(tiny_control_list)
head(tiny_control_list)

In [ ]:
control_used_in_tiny <- tiny_control_list %>% select(Studienummer, update_pregnancy_week_of_sampling, group)
head(control_used_in_tiny )
dim(control_used_in_tiny )

In [ ]:
control_used_in_tiny <- control_used_in_tiny %>% droplevels()
table(control_used_in_tiny$group)

In [ ]:
tiny_cases <- matched_data_tiny %>% filter(group == "Case_tiny")
dim(tiny_cases)

In [ ]:
used_controls <- c()
for (i in 1:nrow(tiny_cases)) {
  case_row <- tiny_cases[i, ]
  free_controls <- tiny_control_list %>% 
    filter(!(Studienummer %in% used_controls))
  
  if (nrow(free_controls) < 2) {
    message("Not enough available controls for case: ", case_row$Studienummer)
    break
  }
    free_controls$case_distance <- abs(free_controls$distance - case_row$distance)
  
  closest_controls <- free_controls %>%
    arrange(case_distance) %>%
    slice(1:2)

  tiny_control  <- rbind(tiny_control, data.frame(
    Case_tiny = case_row$Studienummer,
    Control1 = closest_controls$Studienummer[1],
    Control2 = closest_controls$Studienummer[2],
    stringsAsFactors = FALSE
  ))
  
  used_controls <- c(used_controls, closest_controls$Studienummer)
}
head(tiny_control)
dim(tiny_control)

In [ ]:
#free_controls
used_controls

In [ ]:
tiny_control <- data.frame(Case_tiny = character(),
                                  Control1 = character(),
                                  Control2 = character(),
                                  stringsAsFactors = FALSE)
head(tiny_control)

##### crosscheck-tiny

In [ ]:
#Cross check the week
case_week_tiny <- matched_data_tiny[matched_data_tiny$Studienummer %in% tiny_control$Case_tiny,]
new_case_week_tiny <- case_week_tiny %>% select(Studienummer,update_pregnancy_week_of_sampling)
# Rename multiple columns for old to new
setnames(new_case_week_tiny, old = c("Studienummer","update_pregnancy_week_of_sampling"), 
         new = c("Case_tiny","Case_week"))
head(new_case_week_tiny)
dim(new_case_week_tiny)

In [ ]:
control2_week_tiny <- matched_data_tiny[matched_data_tiny$Studienummer %in% tiny_control$Control2,]
control2_week_tiny <- control2_week_tiny %>% select(Studienummer,update_pregnancy_week_of_sampling)
dim(control2_week_tiny)
setnames(control2_week_tiny, old = c("Studienummer","update_pregnancy_week_of_sampling"), 
         new = c("Control2","Control2_week"))
head(control2_week_tiny)
dim(control2_week_tiny)

In [ ]:
control1_week_tiny <- matched_data_tiny[matched_data_tiny$Studienummer %in% tiny_control$Control1,]
control1_week_tiny <- control1_week_tiny %>% select(Studienummer,update_pregnancy_week_of_sampling)
# Rename multiple columns for old to new
setnames(control1_week_tiny, old = c("Studienummer","update_pregnancy_week_of_sampling"), 
         new = c("Control1","Control1_week"))
head(control1_week_tiny )
dim(control1_week_tiny)

In [ ]:
merge_1_tiny <- merge(tiny_control,new_case_week_tiny, by = "Case_tiny", all.x = TRUE) 
head(merge_1_tiny)
dim(merge_1_tiny)

In [ ]:
merge_2_tiny <- merge(merge_1_tiny,control1_week_tiny, by = "Control1", all.x = TRUE) 
head(merge_2_tiny)
dim(merge_2_tiny)

In [ ]:
merge_3_tiny <- merge(merge_2_tiny,control2_week_tiny, by = "Control2", all.x = TRUE) 
head(merge_3_tiny)
dim(merge_3_tiny)

In [ ]:
only_week_tiny <- merge_3_tiny %>% select(Case_week,Control1_week, Control2_week)
head(only_week_tiny)
only_week_tiny
#week is matching

#### Small

In [ ]:
# Limit Case_small to 171 samples
case_small_limit <- new_raw_data %>% filter(group == "Case_small") %>% slice_sample(n = 171)
head(case_small_limit)

In [ ]:
table(case_small_limit$group)

In [ ]:
case_small_limit <- case_small_limit %>% droplevels()
table(case_small_limit$group)

In [ ]:
data_small_control <- bind_rows(control_used_in_tiny, case_small_limit)
head(data_small_control)
dim(data_small_control)
table(data_small_control$group)

#data_small_control

In [ ]:
set.seed(42) 
m.out_small_control <- matchit(group ~ update_pregnancy_week_of_sampling, 
                              data = data_small_control, 
                              method = "nearest", 
                              ratio = 2)
matched_data_small <- match.data(m.out_small_control)
dim(matched_data_small)
#matched_data_small

In [ ]:
small_control_list <- matched_data_small %>% filter(group == "Control")
#small_control_list
table(small_control_list$group)
small_control_list <- small_control_list  %>% droplevels()
table(small_control_list$group)


In [ ]:
#matched_data_small

In [ ]:
small_control <- data.frame(Case_small = character(),
                                  Control1 = character(),
                                  Control2 = character(),
                                  stringsAsFactors = FALSE)
head(small_control)

In [ ]:
small_cases <- matched_data_small %>% filter(group == "Case_small")
dim(small_cases)
table(small_cases$group)

In [ ]:
small_cases <- small_cases %>% droplevels()
table(small_cases$group)


In [ ]:
head(small_cases)

In [ ]:
used_controls <- c()
for (i in 1:nrow(small_cases)) {
  case_row <- small_cases [i, ]
  free_controls <- small_control_list %>% 
    filter(!(Studienummer %in% used_controls))
  
  if (nrow(free_controls) < 2) {
    message("Not enough available controls for case: ", case_row$Studienummer)
    break
  }
    free_controls$case_distance <- abs(free_controls$distance - case_row$distance)
  
  closest_controls <- free_controls %>%
    arrange(case_distance) %>%
    slice(1:2)

 small_control  <- rbind(small_control , data.frame(
    Case_small = case_row$Studienummer,
    Control1 = closest_controls$Studienummer[1],
    Control2 = closest_controls$Studienummer[2],
    stringsAsFactors = FALSE
  ))
  
  used_controls <- c(used_controls, closest_controls$Studienummer)
}
head(small_control)
dim(small_control)

In [ ]:
sum(is.na(small_control))

##### cross check - small

In [ ]:
case_week_small <- matched_data_small[matched_data_small$Studienummer %in% small_control$Case_small,]
new_case_week_small <- case_week_small %>% select(Studienummer,update_pregnancy_week_of_sampling)
# Rename multiple columns for old to new
setnames(new_case_week_small, old = c("Studienummer","update_pregnancy_week_of_sampling"), 
         new = c("Case_small","Case_week"))
head(new_case_week_small)
dim(new_case_week_small)


In [ ]:
control2_week_small <- matched_data_small[matched_data_small$Studienummer %in% small_control$Control2,]
dim(control2_week_small)

In [ ]:
control2_week_small <- matched_data_small[matched_data_small$Studienummer %in% small_control$Control2,]
control2_week_small <- control2_week_small %>% select(Studienummer,update_pregnancy_week_of_sampling)
# Rename multiple columns for old to new
setnames(control2_week_small, old = c("Studienummer","update_pregnancy_week_of_sampling"), 
         new = c("Control2","Control2_week"))
head(control2_week_small)
dim(control2_week_small)

In [ ]:
control1_week_small <-matched_data_small[matched_data_small$Studienummer %in% small_control$Control1,]
control1_week_small <- control1_week_small %>% select(Studienummer,update_pregnancy_week_of_sampling)
# Rename multiple columns for old to new
setnames(control1_week_small, old = c("Studienummer","update_pregnancy_week_of_sampling"), 
         new = c("Control1","Control1_week"))
head(control1_week_small )
dim(control1_week_small)

In [ ]:
merge_1_small <- merge(small_control,new_case_week_small, by = "Case_small", all.x = TRUE) 
head(merge_1_small)
dim(merge_1_small)

In [ ]:
merge_2_small <- merge(merge_1_small,control1_week_small, by = "Control1", all.x = TRUE) 
head(merge_2_small)
dim(merge_2_small)

In [ ]:
merge_3_small <- merge(merge_2_small,control2_week_small, by = "Control2", all.x = TRUE) 
head(merge_3_small)
dim(merge_3_small)

In [ ]:
only_week_small <- merge_3_small %>% select(Case_week,Control1_week, Control2_week)
head(only_week_small )
#only_week_small
#not eaxcat match is observed. for some it is +1 or -1(very few). 

#### Crosscheck- overall 

In [ ]:
#cross checking whether both case and samll have same set of control group.:
intersect(small_control$Case_small, tiny_control$Case_tiny)

In [ ]:
length(intersect(tiny_control$Control1, small_control$Control1))
length(intersect(tiny_control$Control2, small_control$Control2))
length(intersect(tiny_control$Control1, small_control$Control2))
length(intersect(tiny_control$Control2, small_control$Control1))

In [ ]:
length(intersect(tiny_control$Control1, tiny_control$Control2))
length(intersect(small_control$Control1, small_control$Control2))

In [ ]:
tiny_control_all <- c(tiny_control$Control1, tiny_control$Control2)
#tiny_control_all
small_control_all <- c(small_control$Control1, small_control$Control2)

In [ ]:
length(intersect(tiny_control_all , small_control_all))

#### Save the case control list


In [ ]:
write.table(small_control,"data",sep= "\t", row.names = FALSE)


In [ ]:
write.table(tiny_control,"data",
            sep= "\t", row.names = FALSE)

In [ ]:
small_control_excel <- read.table("data",header = TRUE)
head(small_control_excel)

# Write the data to an Excel file
write_xlsx(small_control_excel, "data")

In [ ]:
tiny_control_excel <- read.table("data",header = TRUE)
head(tiny_control_excel)

# Write the data to an Excel file
write_xlsx(tiny_control_excel, "data")

In [ ]:

small_control_excel <- read.table("/ceph/projects/010_SweMaMi/anusha.antony/Anu_SweMaMi/project_neonate_size/SGA_new/SGA_solve_outliers_250102/data/meta/Small_control_matchit_171_samecontrol_250108.tsv",header = TRUE)
head(small_control_excel)

# Write the data to an Excel file
write_xlsx(small_control_excel, "/ceph/projects/010_SweMaMi/anusha.antony/Anu_SweMaMi/project_neonate_size/SGA_new/SGA_solve_outliers_250102/data/meta/Small_control_matchit_171_samecontrol_250109.xlsx")

In [ ]:
tiny_control_excel <- read.table("/ceph/projects/010_SweMaMi/anusha.antony/Anu_SweMaMi/project_neonate_size/SGA_new/SGA_solve_outliers_250102/data/meta/tiny_control_matchit_171_same_control_250108.tsv",header = TRUE)
head(tiny_control_excel)

# Write the data to an Excel file
write_xlsx(tiny_control_excel, "/ceph/projects/010_SweMaMi/anusha.antony/Anu_SweMaMi/project_neonate_size/SGA_new/SGA_solve_outliers_250102/data/meta/tiny_control_matchit_171_same_control_250109.xlsx")